In [7]:
import os
import shutil
import subprocess

import mrcfile
import numpy as np
from scipy.ndimage import zoom
import json
import os
import shutil
import numpy as np

import pandas as pd

In [2]:
def download_map_model(emdb, pdb, resolution, directory):
    emdb_id = emdb.split("-")[1]
    directory1 = f"{directory}/{emdb}_re_{resolution}"
    os.makedirs(directory1, exist_ok=True)
    emdb_fetch_link = f"https://ftp.ebi.ac.uk/pub/databases/emdb/structures/{emdb}/map/emd_{emdb_id}.map.gz"
    pdb_fetch_link = f"http://files.rcsb.org/download/{pdb}.cif"
    if os.path.exists(f"{directory1}/emd_{emdb_id}.map"):
        pass
    else:
        subprocess.run(["wget", "-P", directory1, emdb_fetch_link])
        subprocess.run(["gzip", "-d", f"{directory1}/emd_{emdb_id}.map.gz"])
    if os.path.exists(f"{directory1}/{pdb}.cif"):
        pass
    else:
        subprocess.run(["wget", "-P", directory1, pdb_fetch_link])
        print(f"=> {emdb} and pdb-{pdb} files are downloaded.")

In [3]:
def map_normalizing(map_path):
    with mrcfile.mmap(map_path) as mrc:
        # Load map data
        map_data = np.array(mrc.data, dtype=np.float32)

        # Resample map to 1.0A*1.0A*1.0A grid size
        zoom_factors = [mrc.voxel_size.z, mrc.voxel_size.y, mrc.voxel_size.x]
        map_data = zoom(map_data, zoom_factors)

        # Normalize map values to the range (0.0, 1.0)
        data_99_9 = np.percentile(map_data, 99.9)
        if data_99_9 == 0.:
            print('data_99_9 == 0!!')
            raise ValueError('99.9th percentile of map data is zero')
        map_data /= data_99_9
        map_data = np.clip(map_data, 0., 1.)

    return map_data

In [4]:
def map_output(input_map, map_data, output_map, is_model=False):
    if os.path.exists(output_map):
        os.remove(output_map)

    print(f"=> Writing new map to {output_map}")
    shutil.copyfile(input_map, output_map)
    with mrcfile.open(output_map, mode='r+') as mrc:
        if is_model:
            map_data = map_data.astype(np.int8)
        else:
            map_data = map_data.astype(np.float32)

        mrc.set_data(map_data)
        mrc.header.mz = map_data.shape[0]
        mrc.header.ispg = 1  #401
        mrc.print_header()
        print("=> New map written successfully.")

In [5]:
DOWNLOAD = True
NORMALIZATION = True
MAP2NPY = False

In [23]:
MODEL_PARTS = ['secondary_strctures', 'key_atoms', 'residue_types']

MAIN_HOME_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net'
PATH_SETTINGS = {
    "Filtered_Dateset": [
        os.path.join(MAIN_HOME_PATH, 'Filtered_Dateset', 'Raw'),
        os.path.join(MAIN_HOME_PATH, 'Filtered_Dateset', 'Training'),
        os.path.join(MAIN_HOME_PATH, 'Filtered_Dateset', 'Raw',
                     'final-20240212.csv'),
    ],
}

PATH_KEYS = "Filtered_Dateset"
DATA_PATH, HOME_PATH, csv_path = PATH_SETTINGS[PATH_KEYS]
temp_sample_path = os.path.join(HOME_PATH, "ready_to_train_and_val")
os.makedirs(temp_sample_path, exist_ok=True)


In [24]:
# Read map list and generate raw_map and model downloading paths
df = pd.read_csv(csv_path)
emdbs, pdbs = df["emdb_id"], df["fitted_pdbs"]
resolutions = df["resolution"].astype(str)
emdb_ids = [emdb.split("-")[1] for emdb in emdbs]
folders = [
    f"{emdb}_re_{resolution}" for emdb, resolution in zip(emdbs, resolutions)
]
raw_maps = [f"emd_{emdb_id}.map" for emdb_id in emdb_ids]
models = [f"{pdb}.cif" for pdb in pdbs]
raw_map_paths = [
    f"{DATA_PATH}/{folder}/{raw_map}"
    for folder, raw_map in zip(folders, raw_maps)
]
model_paths = [
    f"{DATA_PATH}/{folder}/{model}" for folder, model in zip(folders, models)
]

In [25]:
map_paths = [
    f"{raw_map_path.split('.map')[0]}_normalized.mrc"
    for raw_map_path in raw_map_paths
]
for idx, emdb_id in enumerate(emdb_ids):
    if DOWNLOAD:
        download_map_model(emdb=emdbs[idx],
                           pdb=pdbs[idx],
                           resolution=resolutions[idx],
                           directory=DATA_PATH)
        # if idx == 20:   # for download limited number of testing data
        #     exit()
    if NORMALIZATION and not os.path.exists(map_paths[idx]):
        map_data = map_normalizing(raw_map_paths[idx])
        map_output(raw_map_paths[idx],
                   map_data,
                   map_paths[idx],
                   is_model=False)

--2024-02-12 13:28:38--  https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-0128/map/emd_0128.map.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5072419 (4.8M) [application/x-gzip]
Saving to: ‘/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-0128_re_3.8/emd_0128.map.gz’

     0K .......... .......... .......... .......... ..........  1%  346K 14s
    50K .......... .......... .......... .......... ..........  2%  173K 21s
   100K .......... .......... .......... .......... ..........  3%  346K 19s
   150K .......... .......... .......... .......... ..........  4%  161M 14s
   200K .......... .......... .......... .......... ..........  5%  346K 14s
   250K .......... .......... .......... .......... ..........  6%  345K 13s
   300K .......... .......... .......... .......... ..........  7%  346K 13s
   350K ......

=> EMD-0128 and pdb-6h25 files are downloaded.
=> Writing new map to /home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-0128_re_3.8/emd_0128_normalized.mrc
nx              : 338
ny              : 338
nz              : 338
mode            : 2
nxstart         : 0
nystart         : 0
nzstart         : 0
mx              : 338
my              : 338
mz              : 338
cella           : (337.5, 337.5, 337.5)
cellb           : (90., 90., 90.)
mapc            : 1
mapr            : 2
maps            : 3
dmin            : 0.0
dmax            : 1.0
dmean           : 0.004283635411411524
ispg            : 1
nsymbt          : 0
extra1          : b'\x00\x00\x00\x00\x00\x00\x00\x00'
exttyp          : b''
nversion        : 0
extra2          : b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

--2024-02-12 13:28:56--  https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-0379/map/emd_0379.map.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12451953 (12M) [application/x-gzip]
Saving to: ‘/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-0379_re_3.7/emd_0379.map.gz’

     0K .......... .......... .......... .......... ..........  0%  345K 35s
    50K .......... .......... .......... .......... ..........  0%  173K 52s
   100K .......... .......... .......... .......... ..........  1%  290M 35s
   150K .......... .......... .......... .......... ..........  1%  345K 35s
   200K .......... .......... .......... .......... ..........  2%  344K 35s
   250K .......... .......... .......... .......... ..........  2%  346K 34s
   300K .......... .......... .......... .......... ..........  2%  347K 34s
   350K ......

=> EMD-0379 and pdb-6n9u files are downloaded.
=> Writing new map to /home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-0379_re_3.7/emd_0379_normalized.mrc
nx              : 344
ny              : 344
nz              : 344
mode            : 2
nxstart         : 0
nystart         : 0
nzstart         : 0
mx              : 344
my              : 344
mz              : 344
cella           : (344., 344., 344.)
cellb           : (90., 90., 90.)
mapc            : 1
mapr            : 2
maps            : 3
dmin            : 0.0
dmax            : 1.0
dmean           : 0.0022457782179117203
ispg            : 1
nsymbt          : 0
extra1          : b'\x00\x00\x00\x00\x00\x00\x00\x00'
exttyp          : b''
nversion        : 0
extra2          : b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00

--2024-02-12 13:29:32--  https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-0874/map/emd_0874.map.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8918863 (8.5M) [application/x-gzip]
Saving to: ‘/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-0874_re_3.69/emd_0874.map.gz’

     0K .......... .......... .......... .......... ..........  0%  341K 25s
    50K .......... .......... .......... .......... ..........  1%  173K 38s
   100K .......... .......... .......... .......... ..........  1%  346K 33s
   150K .......... .......... .......... .......... ..........  2%  201M 25s
   200K .......... .......... .......... .......... ..........  2%  345K 25s
   250K .......... .......... .......... .......... ..........  3%  347K 24s
   300K .......... .......... .......... .......... ..........  4%  346K 24s
   350K .....

=> EMD-0874 and pdb-6ldi files are downloaded.
=> Writing new map to /home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-0874_re_3.69/emd_0874_normalized.mrc
nx              : 392
ny              : 392
nz              : 392
mode            : 2
nxstart         : 0
nystart         : 0
nzstart         : 0
mx              : 392
my              : 392
mz              : 392
cella           : (392.1, 392.1, 392.1)
cellb           : (90., 90., 90.)
mapc            : 1
mapr            : 2
maps            : 3
dmin            : 0.0
dmax            : 1.0
dmean           : 0.005190643947571516
ispg            : 1
nsymbt          : 0
extra1          : b'\x00\x00\x00\x00\x00\x00\x00\x00'
exttyp          : b''
nversion        : 0
extra2          : b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

... (22 outputs left out)

--2024-02-12 14:08:46--  https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-11736/map/emd_11736.map.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 118945634 (113M) [application/x-gzip]
Saving to: ‘/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-11736_re_2.8/emd_11736.map.gz’

     0K .......... .......... .......... .......... ..........  0%  344K 5m38s
    50K .......... .......... .......... .......... ..........  0%  172K 8m25s
   100K .......... .......... .......... .......... ..........  0%  346K 7m29s
   150K .......... .......... .......... .......... ..........  0%  207M 5m36s
   200K .......... .......... .......... .......... ..........  0%  346K 5m36s
   250K .......... .......... .......... .......... ..........  0%  346K 5m36s
   300K .......... .......... .......... .......... ..........  0%  345K 

=> EMD-11736 and pdb-7ae1 files are downloaded.
=> Writing new map to /home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-11736_re_2.8/emd_11736_normalized.mrc
nx              : 353
ny              : 353
nz              : 353
mode            : 2
nxstart         : 0
nystart         : 0
nzstart         : 0
mx              : 353
my              : 353
mz              : 353
cella           : (352.8, 352.8, 352.8)
cellb           : (90., 90., 90.)
mapc            : 1
mapr            : 2
maps            : 3
dmin            : 0.0
dmax            : 1.0
dmean           : 0.014294458553195
ispg            : 1
nsymbt          : 0
extra1          : b'\x00\x00\x00\x00\x00\x00\x00\x00'
exttyp          : b''
nversion        : 0
extra2          : b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

--2024-02-12 14:13:23--  https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-11842/map/emd_11842.map.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 62612109 (60M) [application/x-gzip]
Saving to: ‘/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-11842_re_3.9/emd_11842.map.gz’

     0K .......... .......... .......... .......... ..........  0%  345K 2m57s
    50K .......... .......... .......... .......... ..........  0%  345K 2m57s
   100K .......... .......... .......... .......... ..........  0%  346K 2m57s
   150K .......... .......... .......... .......... ..........  0%  345K 2m57s
   200K .......... .......... .......... .......... ..........  0%  345K 2m56s
   250K .......... .......... .......... .......... ..........  0%  346K 2m56s
   300K .......... .......... .......... .......... ..........  0%  165M 2m

KeyboardInterrupt: 